# 🌕 The Pareidolia Paradox — FINAL notebook (v3)

### What your last run told us
* OOF balanced accuracy was **0.741**, and validation *peaked at epoch 4–5 then collapsed* while train loss kept falling → the network was memorising, not learning a usable cue.
* The §5 physics table was the smoking gun: rotating by **−az** and by **+az** gave *almost the same* lighting-dipole strength (29.3 vs 31.1) pointing in *mirrored* directions. If a single convention were true, only one sign could be strong.
  The consistent explanation: the data mixes **two sun-angle conventions**, so "rotate by −az" aligns ≈ half the images and scrambles the rest. Half right (~95 %) + half random (~50 %) ≈ **72–75 %** — exactly what you measured.

### What this version does differently
1. **Measures the azimuth↔shading relation per image** (brightness-dipole vector, label-resolved fit `light = φ0 + a·az`) instead of a class-mean heuristic, and reports whether **one or both** conventions are present.
2. **Compares input representations empirically** in a ~10-minute pilot — `spec` (−az), `opp` (+az), `dual` (both + raw as 3 channels), `routed` (per-image convention chosen from the image's own shading) — and uses the winner. If the mixture theory is right, `routed`/`dual` should jump well above the ~0.74 you had.
3. Training regime fixed for the overfitting you saw: fewer epochs (10), stronger regularisation (drop-path, weight-decay 0.05, label smoothing 0.1), patience 4, 320 px.
4. Same code as the GitHub files (`physics.py`, `common.py`) — this notebook writes them to disk, so notebook and repo can't drift apart.

> ⚠️ **Honest limits:** I couldn't run this on your data/GPU. The physics module was tested on synthetic data (single-convention *and* mixture cases); the PyTorch training code was only syntax-checked. **The §6 pilot table and the §8 OOF score are the real verdict.** If they're still < 0.95, send me the §5 + §6 output and we'll iterate.

**Runtime:** GPU (T4 ok). Needs `train_images.zip`, `test_images.zip`, `train_metadata.csv`, `test_metadata.csv` in `MyDrive/Pareidolia Paradox/`.

## 1 · Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q timm albumentations opencv-python-headless scikit-learn h5py

### Shared code (identical to `physics.py` / `common.py` in the GitHub repo)

In [ ]:
%%writefile physics.py
"""physics.py - illumination handling for The Pareidolia Paradox (numpy + OpenCV only).

Background
----------
A crater and a mound look identical when the light direction is unknown (pareidolia).  The sun azimuth tells us where
the light comes from, so the task is to rotate every crop so the light always arrives from the same direction.

The competition text says "rotate CCW by -sun_azimuth_angle".  Rather than trusting that blindly, this module
*measures* how the shading direction in the images relates to the azimuth column:

  * light_moments()  : per-image "brightness dipole" vector (points toward the bright side of the object).
                       A mound is bright on the sun side, a crater on the far side, so the vector is +/- the light dir.
  * probe_light_model(): using the training labels, fits   light_angle = phi0 + a * azimuth   for a in {-2..2}
                       (a=+1 -> spec rule "-az" is right, a=-1 -> the opposite sign, both strong -> a MIXTURE of conventions).
  * route_hypotheses(): label-free per-image choice between a=+1 and a=-1 (works on test images too).
  * build_inputs()   : produces the network input for a chosen mode: spec | opp | dual | routed.

All angles are "visual": measured counter-clockwise from +x with y pointing UP.
"""
import math
import numpy as np
import cv2

MODES = ("spec", "opp", "dual", "routed")


# --------------------------------------------------------------------------------------- rotation
def rotate_ccw(img, deg):
    """Rotate a 2-D uint8 image counter-clockwise by `deg` degrees (reflect-pad -> rotate -> centre-crop; no black corners)."""
    h, w = img.shape[:2]
    diag = int(math.ceil(math.hypot(h, w)))
    ph, pw = (diag - h) // 2 + 4, (diag - w) // 2 + 4
    padded = cv2.copyMakeBorder(img, ph, ph, pw, pw, cv2.BORDER_REFLECT_101)
    H, W = padded.shape[:2]
    M = cv2.getRotationMatrix2D(((W - 1) / 2.0, (H - 1) / 2.0), float(deg), 1.0)   # +deg = counter-clockwise
    rot = cv2.warpAffine(padded, M, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
    return rot[ph:ph + h, pw:pw + w]


def rotate_all(raw, degs):
    degs = np.broadcast_to(np.asarray(degs, dtype=np.float64), (len(raw),))
    return np.stack([rotate_ccw(raw[i], degs[i]) for i in range(len(raw))])


# --------------------------------------------------------------------------------------- measuring the light
def light_moments(raw, blur_sigma=24.0, win_sigma=70.0):
    """Per-image brightness-dipole vector (mx, my) in visual coords (x right, y up).
    High-passed (removes ramps), Gaussian-windowed around the centre object.  Returns float32 (N, 2)."""
    S = raw.shape[1]
    ys, xs = np.mgrid[0:S, 0:S]
    X = (xs - (S - 1) / 2.0).astype(np.float32)
    Y = -(ys - (S - 1) / 2.0).astype(np.float32)
    w = np.exp(-(X ** 2 + Y ** 2) / (2 * win_sigma ** 2)).astype(np.float32)
    out = np.empty((len(raw), 2), np.float32)
    for i, im in enumerate(raw):
        f = im.astype(np.float32)
        hp = (f - cv2.GaussianBlur(f, (0, 0), blur_sigma)) * w
        out[i] = ((hp * X).sum(), (hp * Y).sum())
    return out


def _weights(m):
    mag = np.hypot(m[:, 0], m[:, 1])
    return np.clip(mag / (np.median(mag) + 1e-9), 0, 3)


def probe_light_model(m, az_deg, y, n_null=5, seed=0):
    """Fit  light_angle = phi0 + a*azimuth  for a in {-2,-1,0,1,2} using labels to resolve the crater/mound 180-deg ambiguity.
    Returns {a: {"R": resultant length (0..1), "phi0": radians, "R_null": same statistic with shuffled azimuth}}."""
    theta = np.arctan2(m[:, 1], m[:, 0])
    psi = theta + np.pi * (1 - np.asarray(y))            # mound -> +light dir, crater -> flipped back by pi
    azr = np.deg2rad(np.asarray(az_deg, dtype=np.float64))
    wt = _weights(m); rng = np.random.default_rng(seed)
    res = {}
    for a in (-2, -1, 0, 1, 2):
        c = np.sum(wt * np.exp(1j * (psi - a * azr))) / wt.sum()
        nulls = [abs(np.sum(wt * np.exp(1j * (psi - a * azr[rng.permutation(len(azr))]))) / wt.sum()) for _ in range(n_null)]
        res[a] = {"R": float(abs(c)), "phi0": float(np.angle(c)), "R_null": float(np.mean(nulls))}
    return res


def route_hypotheses(m, az_deg, phi0_deg):
    """Label-free routing: for each image pick a in {+1,-1} whose predicted light AXIS best matches the measured one.
    phi0_deg = {1: deg, -1: deg}.  Returns (a_per_image, score_plus, score_minus)."""
    theta = np.arctan2(m[:, 1], m[:, 0]); azr = np.deg2rad(np.asarray(az_deg, dtype=np.float64))
    s = {a: np.cos(2 * (theta - np.deg2rad(phi0_deg[a]) - a * azr)) for a in (1, -1)}
    return np.where(s[1] >= s[-1], 1, -1), s[1], s[-1]


def discover_physics(raw, az_deg, y, seed=0):
    """Run the whole probe on the training set.  Returns a dict that is JSON-friendly except for 'm' (per-image moments)."""
    m = light_moments(raw)
    fit = probe_light_model(m, az_deg, y, seed=seed)
    phi0 = {1: math.degrees(fit[1]["phi0"]), -1: math.degrees(fit[-1]["phi0"])}
    theta = np.arctan2(m[:, 1], m[:, 0]); psi = theta + np.pi * (1 - np.asarray(y)); wt = _weights(m)

    def routed_R(az_used):
        a_i, _, _ = route_hypotheses(m, az_used, phi0)
        azr = np.deg2rad(az_used)
        L = np.where(a_i == 1, np.deg2rad(phi0[1]) + azr, np.deg2rad(phi0[-1]) - azr)
        return float(abs(np.sum(wt * np.exp(1j * (psi - L))) / wt.sum())), float(np.mean(a_i == 1))

    R_routed, p_plus = routed_R(np.asarray(az_deg, dtype=np.float64))
    rng = np.random.default_rng(seed + 1)
    R_routed_null = float(np.mean([routed_R(np.asarray(az_deg, dtype=np.float64)[rng.permutation(len(m))])[0] for _ in range(5)]))
    return {"m": m, "fit": fit, "phi0_deg": phi0, "R_routed": R_routed, "R_routed_null": R_routed_null, "p_plus": p_plus}


def summarize_probe(d):
    """Human-readable lines describing the probe result."""
    L = ["hypothesis: light_angle = phi0 + a*azimuth     (a=+1 <=> competition rule 'rotate CCW by -az' is correct)",
         f"{'a':>3} {'R':>8} {'null':>8} {'R/null':>8}"]
    for a in (-2, -1, 0, 1, 2):
        f = d["fit"][a]; L.append(f"{a:>3} {f['R']:>8.3f} {f['R_null']:>8.3f} {f['R'] / max(f['R_null'], 1e-9):>8.1f}")
    L.append(f"per-image routed between a=+1 / a=-1: R={d['R_routed']:.3f} (null {d['R_routed_null']:.3f}), share routed to a=+1: {d['p_plus']:.2f}")
    Rp, Rm = d["fit"][1]["R"], d["fit"][-1]["R"]
    if max(Rp, Rm) < 2 * max(d["fit"][1]["R_null"], d["fit"][-1]["R_null"]):
        L.append("verdict: weak/unclear relation between azimuth and measured shading (estimator is crude) -> rely on the pilot below.")
    elif min(Rp, Rm) > 0.5 * max(Rp, Rm) and d["R_routed"] > 1.2 * max(Rp, Rm):
        L.append("verdict: BOTH conventions present (mixture) -> 'routed' / 'dual' modes should win.")
    else:
        L.append(f"verdict: a single convention dominates ({'spec rule (-az)' if Rp >= Rm else 'opposite sign (+az)'}).")
    return L


# --------------------------------------------------------------------------------------- network inputs
def build_inputs(raw, az_deg, mode, phi0_deg, m=None):
    """Return uint8 array (N, H, W, C).  Every mode also adds a constant offset so the light ends up at angle 0 (from +x/right)
    under its hypothesis; a constant rotation is harmless for the CNN but makes the vertical flip label-preserving.
      spec   : rotate CCW by -(az + phi0[+1])           (competition rule)
      opp    : rotate CCW by -(-az + phi0[-1])          (opposite sign)
      dual   : channels [spec, opp, raw]                (network decides; safest fallback)
      routed : per image spec or opp, chosen from the image's own shading axis (needs `m`)"""
    az = np.asarray(az_deg, dtype=np.float64)
    d = lambda a: -(a * az + phi0_deg[a])
    if mode == "spec":
        return rotate_all(raw, d(1))[..., None]
    if mode == "opp":
        return rotate_all(raw, d(-1))[..., None]
    if mode == "dual":
        return np.stack([rotate_all(raw, d(1)), rotate_all(raw, d(-1)), raw], axis=-1)
    if mode == "routed":
        if m is None: m = light_moments(raw)
        a_i, _, _ = route_hypotheses(m, az, phi0_deg)
        return rotate_all(raw, np.where(a_i == 1, d(1), d(-1)))[..., None]
    raise ValueError(f"unknown mode {mode!r}; choose from {MODES}")

In [ ]:
%%writefile common.py
"""common.py - data loading, model, training and inference utilities (shared by train.py, inference.py and the Colab notebook)."""
import os, json, math, time, random, copy, zipfile
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import balanced_accuracy_score, confusion_matrix  # noqa: F401 (re-exported)
from tqdm.auto import tqdm

from physics import *   # rotate_ccw, light_moments, discover_physics, build_inputs, ...

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


# ======================================================================================= data
def find_image_root(root, sample_name):
    for dp, _, files in os.walk(root):
        if sample_name in files:
            return dp
    raise FileNotFoundError(f"{sample_name} not found under {root}")


def _image_dir(data_dir, work_dir, name, sample):
    plain = os.path.join(data_dir, name)
    if os.path.isdir(plain) and os.listdir(plain):
        return find_image_root(plain, sample)
    zpath = os.path.join(data_dir, name + ".zip")
    if not os.path.exists(zpath):
        raise FileNotFoundError(f"Need {zpath} (or an extracted folder {plain})")
    out = os.path.join(work_dir, name)
    if not os.path.isdir(out) or not os.listdir(out):
        print(f"Extracting {name}.zip ..."); os.makedirs(out, exist_ok=True)
        with zipfile.ZipFile(zpath) as z:
            z.extractall(out)
    return find_image_root(out, sample)


def load_gray_stack(df, img_dir, size=256):
    arr = np.empty((len(df), size, size), np.uint8)
    for i, iid in enumerate(tqdm(df["image_id"].values, desc=f"loading {os.path.basename(img_dir)}")):
        im = cv2.imread(os.path.join(img_dir, iid), cv2.IMREAD_GRAYSCALE)
        if im is None:
            raise FileNotFoundError(f"Unreadable image: {iid}")
        if im.shape != (size, size):
            im = cv2.resize(im, (size, size), interpolation=cv2.INTER_AREA)
        arr[i] = im
    return arr


def load_data(data_dir, work_dir="/tmp/pareidolia_work", need_train=True):
    """Returns (train_meta, RAW_TR, test_meta, RAW_TE); train parts are None when need_train=False."""
    os.makedirs(work_dir, exist_ok=True)
    test_meta = pd.read_csv(os.path.join(data_dir, "test_metadata.csv"))
    assert {"image_id", "sun_azimuth_angle"} <= set(test_meta.columns) and test_meta["sun_azimuth_angle"].notna().all()
    raw_te = load_gray_stack(test_meta, _image_dir(data_dir, work_dir, "test_images", test_meta.image_id.iloc[0]))
    if not need_train:
        return None, None, test_meta, raw_te
    train_meta = pd.read_csv(os.path.join(data_dir, "train_metadata.csv"))
    assert {"image_id", "label", "sun_azimuth_angle"} <= set(train_meta.columns) and train_meta["sun_azimuth_angle"].notna().all()
    raw_tr = load_gray_stack(train_meta, _image_dir(data_dir, work_dir, "train_images", train_meta.image_id.iloc[0]))
    return train_meta, raw_tr, test_meta, raw_te


# ======================================================================================= dataset / augmentation
def make_transforms(img_size, flip):
    """`flip` = vertical flip.  It is label-preserving ONLY because build_inputs() puts the light at angle 0 (from the right):
    mirroring top<->bottom leaves the light direction unchanged.  Never flip left<->right or rotate by 90/180 deg."""
    S = img_size
    aug = [A.Resize(S, S)]
    if flip:
        aug.append(A.VerticalFlip(p=0.5))
    aug += [A.Affine(translate_percent=0.03, scale=(0.92, 1.08), rotate=0, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
            A.GaussNoise(p=0.2), A.CoarseDropout(p=0.15),
            A.Normalize(mean=MEAN, std=STD), ToTensorV2()]
    base = [A.Resize(S, S), A.Normalize(mean=MEAN, std=STD), ToTensorV2()]
    tta = [A.Compose(base)]
    if flip:
        tta.append(A.Compose([A.Resize(S, S), A.VerticalFlip(p=1.0), A.Normalize(mean=MEAN, std=STD), ToTensorV2()]))
    return A.Compose(aug), A.Compose(base), tta


class LunarDataset(Dataset):
    """imgs: uint8 (N,H,W,C) with C = 1 (replicated to 3) or 3."""
    def __init__(self, imgs, labels, transform):
        self.imgs, self.labels, self.transform = imgs, labels, transform
    def __len__(self):
        return len(self.imgs)
    def __getitem__(self, i):
        im = self.imgs[i]
        if im.shape[-1] == 1:
            im = np.repeat(im, 3, axis=2)
        x = self.transform(image=np.ascontiguousarray(im))["image"]
        return x if self.labels is None else (x, torch.tensor(int(self.labels[i]), dtype=torch.long))


# ======================================================================================= model
def build_backbone(name, pretrained=True, drop_path=0.0):
    for kw in ({"drop_path_rate": drop_path}, {}):
        try:
            return timm.create_model(name, pretrained=pretrained, num_classes=0, global_pool="avg", **kw), name
        except TypeError:
            continue
        except Exception as e:
            print(f"Could not load '{name}' ({e}); falling back to resnet50.")
            break
    return timm.create_model("resnet50", pretrained=pretrained, num_classes=0, global_pool="avg"), "resnet50"


class LunarNet(nn.Module):
    def __init__(self, backbone_name, pretrained=True, drop_path=0.0, num_classes=2):
        super().__init__()
        self.backbone, self.backbone_name = build_backbone(backbone_name, pretrained, drop_path)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.backbone.num_features, 256), nn.ReLU(inplace=True),
                                  nn.Dropout(0.2), nn.Linear(256, num_classes))
    def forward(self, x):
        return self.head(self.backbone(x))


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.0):
        super().__init__(); self.alpha, self.gamma, self.ls = alpha, gamma, label_smoothing
    def forward(self, logits, y):
        pt = torch.exp(-F.cross_entropy(logits, y, label_smoothing=self.ls, reduction="none").detach())
        ce = F.cross_entropy(logits, y, weight=self.alpha, label_smoothing=self.ls, reduction="none")
        return (((1 - pt) ** self.gamma) * ce).mean()


def class_weights(y):
    c = np.bincount(y, minlength=2); return torch.tensor(c.sum() / (len(c) * c), dtype=torch.float32)


class EMA:
    """Exponential moving average of the weights (with warm-up)."""
    def __init__(self, model, decay):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay, self.n = decay, 0
    @torch.no_grad()
    def update(self, model):
        self.n += 1; d = min(self.decay, (1 + self.n) / (10 + self.n)); msd = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else:
                v.copy_(msd[k])


@torch.no_grad()
def _evaluate(model, loader, crit, device, amp):
    model.eval(); tot, probs, labels = 0.0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type=device, enabled=amp):
            out = model(x)
        tot += crit(out.float(), y).item() * x.size(0)
        probs.append(F.softmax(out.float(), 1).cpu().numpy()); labels.append(y.cpu().numpy())
    probs, labels = np.concatenate(probs), np.concatenate(labels)
    return tot / len(loader.dataset), balanced_accuracy_score(labels, probs.argmax(1)), probs


# ======================================================================================= training
def fit_model(X_tr, y_tr, X_va, y_va, backbone, img_size=320, epochs=10, lr=1e-4, batch_size=16, flip=False, use_ema=True,
              ema_decay=0.999, weight_decay=0.05, patience=4, gamma=2.0, label_smoothing=0.1, drop_path=0.2,
              head_lr_mult=10.0, grad_clip=1.0, warmup_epochs=1, num_workers=2, pretrained=True, save_path=None,
              meta=None, device=None, log=print):
    """Trains one model. Model selection uses balanced accuracy (the competition metric). Returns dict(best, probs, epochs_run)."""
    device = device or DEVICE; amp = device == "cuda"
    train_tf, val_tf, _ = make_transforms(img_size, flip)
    tr = DataLoader(LunarDataset(X_tr, y_tr, train_tf), batch_size=batch_size, shuffle=True, num_workers=num_workers,
                    pin_memory=amp, drop_last=len(X_tr) >= 2 * batch_size, persistent_workers=num_workers > 0)
    va = DataLoader(LunarDataset(X_va, y_va, val_tf), batch_size=batch_size * 2, shuffle=False, num_workers=num_workers, pin_memory=amp)
    model = LunarNet(backbone, pretrained, drop_path).to(device)
    crit = FocalLoss(class_weights(y_tr).to(device), gamma, label_smoothing)
    opt = torch.optim.AdamW([{"params": list(model.backbone.parameters()), "lr": lr},
                             {"params": list(model.head.parameters()), "lr": lr * head_lr_mult}], weight_decay=weight_decay)
    total, warm = epochs * len(tr), max(1, warmup_epochs * len(tr))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    ema = EMA(model, ema_decay) if use_ema else None
    eval_model = ema.module if ema is not None else model

    best, best_probs, bad, ran = -1.0, None, 0, 0
    for ep in range(epochs):
        t0 = time.time(); model.train(); tot = 0.0
        for x, y in tr:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device, enabled=amp):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(opt); scaler.update(); sched.step()
            if ema is not None:
                ema.update(model)
            tot += loss.item() * x.size(0)
        vl, vb, vp = _evaluate(eval_model, va, crit, device, amp); ran = ep + 1
        log(f"  epoch {ep + 1:>2}/{epochs} | train {tot / len(tr.dataset):.4f} | val_loss {vl:.4f} | val_bal_acc {vb:.4f} | {time.time() - t0:.0f}s")
        if vb > best:
            best, best_probs, bad = vb, vp, 0
            if save_path:
                torch.save({"model_state": eval_model.state_dict(), "backbone": model.backbone_name, "img_size": img_size,
                            "val_bal_acc": vb, "meta": meta or {}}, save_path)
        else:
            bad += 1
            if bad >= patience:
                log(f"  early stop at epoch {ep + 1}"); break
    del model, ema
    if amp:
        torch.cuda.empty_cache()
    return {"best": best, "probs": best_probs, "epochs_run": ran}


def kfold_train(X, y, run_name, out_dir, n_folds=5, folds=None, seed=42, resume=True, log=print, **fit_kwargs):
    os.makedirs(out_dir, exist_ok=True)
    splits = list(StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed).split(np.zeros(len(y)), y))
    scores = {}
    for f in (range(n_folds) if folds is None else folds):
        ckpt, oof = os.path.join(out_dir, f"model_{run_name}_fold{f}.pt"), os.path.join(out_dir, f"oof_{run_name}_fold{f}.npz")
        if resume and os.path.exists(ckpt) and os.path.exists(oof):
            log(f"fold {f}: already finished - skipping"); continue
        log(f"\n===== {run_name} | fold {f} ====="); set_seed(seed + f)
        tr, va = splits[f]
        res = fit_model(X[tr], y[tr], X[va], y[va], save_path=ckpt, log=log, **fit_kwargs)
        np.savez(oof, idx=va, probs=res["probs"]); scores[f] = res["best"]
        log(f"fold {f} best val balanced accuracy: {res['best']:.4f}")
    return scores


def gather_oof(out_dir, runs, n_folds, n):
    probs = np.zeros((n, 2), np.float32); cnt = np.zeros(n)
    for r in runs:
        for f in range(n_folds):
            p = os.path.join(out_dir, f"oof_{r}_fold{f}.npz")
            if os.path.exists(p):
                d = np.load(p); probs[d["idx"]] += d["probs"]; cnt[d["idx"]] += 1
    m = cnt > 0; probs[m] /= cnt[m, None]
    return probs, m


def tune_threshold(y, p1, lo=0.2, hi=0.8):
    """Threshold on P(class 1) maximising balanced accuracy; only adopted if it beats 0.5 by > 0.1 %."""
    ths = np.linspace(lo, hi, 121); accs = np.array([balanced_accuracy_score(y, (p1 >= t).astype(int)) for t in ths])
    acc05 = balanced_accuracy_score(y, (p1 >= 0.5).astype(int)); t = float(ths[accs.argmax()])
    return (t if accs.max() - acc05 > 0.001 else 0.5), float(accs.max()), float(acc05), ths, accs


def pilot(raw, az, y, backbone="efficientnet_b0.ra_in1k", modes=MODES, phi0_deg=None, img_size=224, epochs=5, n_max=5000,
          lr=3e-4, batch_size=32, seed=0, num_workers=2, pretrained=True, log=print):
    """Fast experiment: same subsample / same split / same small model for each candidate input representation."""
    idx = np.arange(len(y))
    if len(idx) > n_max:
        idx, _ = train_test_split(idx, train_size=n_max, stratify=y, random_state=seed)
    tr, va = train_test_split(np.arange(len(idx)), test_size=0.25, stratify=y[idx], random_state=seed)
    m_all = light_moments(raw[idx]); out = {}
    for mode in modes:
        log(f"\n--- pilot mode: {mode} ---")
        X = build_inputs(raw[idx], az[idx], mode, phi0_deg, m=m_all)
        res = fit_model(X[tr], y[idx][tr], X[va], y[idx][va], backbone=backbone, img_size=img_size, epochs=epochs, lr=lr,
                        batch_size=batch_size, flip=False, use_ema=False, drop_path=0.0, weight_decay=1e-2, patience=epochs,
                        num_workers=num_workers, pretrained=pretrained, log=log)
        out[mode] = res["best"]; log(f"pilot {mode}: best val balanced accuracy {res['best']:.4f}")
        del X
    return out


# ======================================================================================= inference
@torch.no_grad()
def predict_probs(model, X, img_size, flip, device=None, batch_size=64, num_workers=2):
    device = device or DEVICE; amp = device == "cuda"; model.eval()
    _, _, tta = make_transforms(img_size, flip); total = np.zeros((len(X), 2), np.float32)
    for tf in tta:
        loader = DataLoader(LunarDataset(X, None, tf), batch_size=batch_size, shuffle=False, num_workers=num_workers)
        outs = []
        for x in loader:
            with torch.autocast(device_type=device, enabled=amp):
                outs.append(F.softmax(model(x.to(device)).float(), 1).cpu().numpy())
        total += np.concatenate(outs)
    return total / len(tta)


def ensemble_predict(weights_dir, runs, X, flip, n_folds=5, device=None, log=print):
    device = device or DEVICE; probs, used = [], []
    for r in runs:
        for f in range(n_folds):
            p = os.path.join(weights_dir, f"model_{r}_fold{f}.pt")
            if not os.path.exists(p):
                continue
            ck = torch.load(p, map_location=device, weights_only=False)
            m = LunarNet(ck["backbone"], pretrained=False).to(device); m.load_state_dict(ck["model_state"])
            log(f"{r} fold {f} (val_bal_acc={ck['val_bal_acc']:.4f}) -> inference")
            probs.append(predict_probs(m, X, ck["img_size"], flip, device)); used.append((r, f)); del m
    assert probs, f"No checkpoints matching model_<run>_fold<k>.pt found in {weights_dir}"
    return np.mean(probs, 0), used


def save_config(out_dir, cfg):
    with open(os.path.join(out_dir, "pipeline_config.json"), "w") as f:
        json.dump(cfg, f, indent=2)


def load_config(out_dir):
    with open(os.path.join(out_dir, "pipeline_config.json")) as f:
        cfg = json.load(f)
    cfg["phi0_deg"] = {int(k): float(v) for k, v in cfg["phi0_deg"].items()}
    return cfg


def make_submission(test_meta, p1, threshold, path):
    sub = pd.DataFrame({"image_id": test_meta["image_id"].values, "label": (p1 >= threshold).astype(int)})
    assert len(sub) == 2000, f"Expected 2000 rows, got {len(sub)}"
    assert list(sub.columns) == ["image_id", "label"] and sub["label"].isin([0, 1]).all() and sub.notna().all().all()
    assert sub["image_id"].is_unique
    sub.to_csv(path, index=False)
    return sub

In [ ]:
import sys; sys.path.insert(0, "/content")
import matplotlib.pyplot as plt
from common import *
print("Torch", torch.__version__, "| timm", timm.__version__, "| albumentations", A.__version__)
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "⚠️ NO GPU — Runtime > Change runtime type > GPU")

## 2 · Config

In [ ]:
class CFG:
    SEED = 42
    PROJECT_DIR = "/content/drive/MyDrive/Pareidolia Paradox"   # data + all outputs live here
    WORK_DIR    = "/content/pareidolia_work"                    # fast local scratch

    MODE = "auto"                    # auto | spec | opp | dual | routed   (auto = pick by pilot)
    PILOT_EPOCHS = 5

    BACKBONE = "tf_efficientnetv2_s.in21k_ft_in1k"   # 2nd run idea: "convnext_tiny.fb_in22k_ft_in1k" (+ RUN_NAME="convnext", LR=5e-5)
    RUN_NAME = "effv2s"
    IMG_SIZE = 320                   # 384/448 if OOF is close to but below 0.95
    EPOCHS, LR, BATCH_SIZE = 10, 1e-4, 16
    N_FOLDS = 5
    NUM_WORKERS = 2
    RESUME = True                    # reuse saved physics config + finished folds after a disconnect

set_seed(CFG.SEED); os.makedirs(CFG.WORK_DIR, exist_ok=True)

## 3 · Load data (read once into RAM)

In [ ]:
train_meta, RAW_TR, test_meta, RAW_TE = load_data(CFG.PROJECT_DIR, CFG.WORK_DIR)
Y_TR  = train_meta["label"].values.astype(int)
AZ_TR = train_meta["sun_azimuth_angle"].values.astype(np.float64)
AZ_TE = test_meta["sun_azimuth_angle"].values.astype(np.float64)
assert len(test_meta) == 2000
print("train cols:", train_meta.columns.tolist(), "| test cols:", test_meta.columns.tolist())
print(RAW_TR.shape, RAW_TE.shape, "| class counts:", np.bincount(Y_TR).tolist(), "| azimuth", AZ_TR.min(), "-", AZ_TR.max())

## 4 · Physics probe — how does the shading direction relate to `sun_azimuth_angle`?

For each image the *brightness-dipole vector* points toward the bright side of the object (sun side for a mound, far side for a crater).
Using the labels we fit `light_angle = φ0 + a·azimuth`:

* **a = +1** → the competition rule "rotate CCW by −az" is right  · **a = −1** → the opposite sign is right
* **both strong (each ≈ half)** → the dataset mixes both conventions

The histograms show the residual `light − a·az − φ0`: a **sharp peak** = that hypothesis explains the images. Routed = each image assigned to whichever hypothesis fits its own shading axis.

In [ ]:
D = discover_physics(RAW_TR, AZ_TR, Y_TR)
print("\n".join(summarize_probe(D)))
print("\nφ0 (deg):", {k: round(v, 1) for k, v in D["phi0_deg"].items()})

m = D["m"]; theta = np.arctan2(m[:, 1], m[:, 0]); psi = theta + np.pi * (1 - Y_TR)
wrap = lambda deg: (deg + 180) % 360 - 180
a_i, _, _ = route_hypotheses(m, AZ_TR, D["phi0_deg"])
L_routed = np.where(a_i == 1, D["phi0_deg"][1] + AZ_TR, D["phi0_deg"][-1] - AZ_TR)
panels = [(f"a=+1  (spec rule)  R={D['fit'][1]['R']:.2f}",  wrap(np.degrees(psi) - AZ_TR - D["phi0_deg"][1])),
          (f"a=−1  (opposite)   R={D['fit'][-1]['R']:.2f}", wrap(np.degrees(psi) + AZ_TR - D["phi0_deg"][-1])),
          (f"routed per image   R={D['R_routed']:.2f}",     wrap(np.degrees(psi) - L_routed))]
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
for ax, (t, r) in zip(axes, panels):
    ax.hist(r, bins=72, range=(-180, 180)); ax.set_title(t, fontsize=10); ax.set_xlabel("residual light angle (deg)")
plt.tight_layout(); plt.show()

In [ ]:
# What each candidate representation looks like (top: crater / bottom: mound). Light should sit on a consistent side in the right mode.
sel = np.concatenate([np.where(Y_TR == 0)[0][:5], np.where(Y_TR == 1)[0][:5]])
views = {"raw": RAW_TR[sel][..., None]}
for mode in ("spec", "opp", "routed"):
    views[mode] = build_inputs(RAW_TR[sel], AZ_TR[sel], mode, D["phi0_deg"], m=D["m"][sel])
fig, axes = plt.subplots(len(views), 10, figsize=(20, 2.1 * len(views)))
for r, (name, arr) in enumerate(views.items()):
    for c in range(10):
        axes[r, c].imshow(arr[c][..., 0], cmap="gray"); axes[r, c].axis("off")
        if c == 0: axes[r, c].set_title(name, fontsize=9, loc="left")
plt.tight_layout(); plt.show()

## 5 · Pilot: which input representation wins? (≈ 1–2 min per mode)

Same 5 000-image subsample, same split, same small EfficientNet-B0 for every mode. **This table decides the physics** — it is a measurement, not an assumption.
Ties go to the competition rule (`spec`). Skipped automatically when a saved `pipeline_config.json` exists (`CFG.RESUME`).

In [ ]:
cfg_path = os.path.join(CFG.PROJECT_DIR, "pipeline_config.json")
if CFG.RESUME and os.path.exists(cfg_path):
    cfg = load_config(CFG.PROJECT_DIR); print(f"↩️ Loaded saved config: mode={cfg['mode']} (delete {cfg_path} to redo the pilot)")
else:
    if CFG.MODE == "auto":
        pilot_res = pilot(RAW_TR, AZ_TR, Y_TR, phi0_deg=D["phi0_deg"], epochs=CFG.PILOT_EPOCHS, num_workers=CFG.NUM_WORKERS)
        print("\n================ PILOT (val balanced accuracy) ================")
        for k, v in sorted(pilot_res.items(), key=lambda kv: -kv[1]): print(f"  {k:<7} {v:.4f}")
        mode = max(pilot_res, key=lambda k: (round(pilot_res[k], 3), k == "spec"))
    else:
        mode, pilot_res = CFG.MODE, {}
    if mode == "routed": strong = D["R_routed"] > 0.3 and D["R_routed"] > 3 * D["R_routed_null"]
    else:                strong = D["fit"][{"spec": 1, "opp": -1}.get(mode, 1)]["R"] > 0.3
    cfg = {"mode": mode, "phi0_deg": {str(k): v for k, v in D["phi0_deg"].items()}, "use_vflip": bool(strong and mode != "dual"),
           "img_size": CFG.IMG_SIZE, "backbone": CFG.BACKBONE, "runs": [CFG.RUN_NAME], "threshold": 0.5,
           "pilot": pilot_res, "probe": summarize_probe(D)}
    save_config(CFG.PROJECT_DIR, cfg); cfg = load_config(CFG.PROJECT_DIR)
print(f"\n>>> Using mode = {cfg['mode']} | vertical-flip aug/TTA = {cfg['use_vflip']}")

## 6 · Stratified K-fold training

Class-weighted focal loss, AdamW, warm-up + cosine, EMA, AMP. Best epoch chosen by **balanced accuracy**. Finished folds are skipped on re-run.

In [ ]:
X_TR = build_inputs(RAW_TR, AZ_TR, cfg["mode"], cfg["phi0_deg"])
print("training tensor:", X_TR.shape, X_TR.dtype)
kfold_train(X_TR, Y_TR, CFG.RUN_NAME, CFG.PROJECT_DIR, n_folds=CFG.N_FOLDS, seed=CFG.SEED, resume=CFG.RESUME,
            backbone=CFG.BACKBONE, img_size=CFG.IMG_SIZE, epochs=CFG.EPOCHS, lr=CFG.LR, batch_size=CFG.BATCH_SIZE,
            flip=cfg["use_vflip"], num_workers=CFG.NUM_WORKERS,
            meta={"mode": cfg["mode"], "phi0_deg": cfg["phi0_deg"], "use_vflip": cfg["use_vflip"]})
if CFG.RUN_NAME not in cfg["runs"]:
    cfg["runs"].append(CFG.RUN_NAME); save_config(CFG.PROJECT_DIR, cfg)      # 2nd backbone joins the ensemble automatically

## 7 · Out-of-fold score & threshold (out-of-fold only — never the test set)

In [ ]:
oof, mask = gather_oof(CFG.PROJECT_DIR, cfg["runs"], CFG.N_FOLDS, len(Y_TR))
assert mask.any(), "No OOF files — train first."
thr, best, acc05, ths, accs = tune_threshold(Y_TR[mask], oof[mask, 1])
print(f"OOF samples: {mask.sum()}/{len(Y_TR)} | runs: {cfg['runs']}")
print(f"OOF balanced accuracy @0.5 : {acc05:.4f}\nOOF balanced accuracy @{thr:.3f}: {best:.4f}  (threshold used: {thr:.3f})")
print(confusion_matrix(Y_TR[mask], (oof[mask, 1] >= thr).astype(int)))
print("🎯 TARGET ≥ 0.95:", "✅ reached" if max(acc05, best) >= 0.95 else "❌ not yet — see §12")
cfg["threshold"], cfg["oof_bal_acc"] = thr, max(best, acc05); save_config(CFG.PROJECT_DIR, cfg)
plt.figure(figsize=(6, 3.4)); plt.plot(ths, accs); plt.axvline(thr, color="r", ls="--"); plt.axvline(.5, color="gray", ls=":")
plt.xlabel("threshold"); plt.ylabel("OOF balanced acc"); plt.show()

## 8 · Test predictions (fold/run ensemble + physics-safe TTA) → `submission.csv`

In [ ]:
cfg = load_config(CFG.PROJECT_DIR)
X_TE = build_inputs(RAW_TE, AZ_TE, cfg["mode"], cfg["phi0_deg"])          # SAME physics normalisation as training
probs, used = ensemble_predict(CFG.PROJECT_DIR, cfg["runs"], X_TE, cfg["use_vflip"], n_folds=CFG.N_FOLDS)
sub = make_submission(test_meta, probs[:, 1], cfg["threshold"], os.path.join(CFG.PROJECT_DIR, "submission.csv"))
share = sub["label"].value_counts(normalize=True).sort_index()
print(f"Ensembled {len(used)} models | mode={cfg['mode']} | threshold={cfg['threshold']:.3f}")
print("Predicted class share:", share.round(3).to_dict(), "| train share:", np.round(np.bincount(Y_TR) / len(Y_TR), 3).tolist())
if share.min() < 0.15: print("⚠️ Predictions heavily skewed to one class — do not submit yet; check §4/§5.")
print("Saved:", os.path.join(CFG.PROJECT_DIR, "submission.csv")); sub.head()

## 9 · Export weights (`.pt` per fold + `.h5` of the best model)

In [ ]:
import h5py, glob
best_ck, best_path = None, None
for p in glob.glob(os.path.join(CFG.PROJECT_DIR, "model_*_fold*.pt")):
    ck = torch.load(p, map_location="cpu", weights_only=False)
    if best_ck is None or ck["val_bal_acc"] > best_ck["val_bal_acc"]: best_ck, best_path = ck, p
h5_path = os.path.join(CFG.PROJECT_DIR, "best_model_weights.h5")
with h5py.File(h5_path, "w") as h:
    for k, v in best_ck["model_state"].items(): h.create_dataset(k, data=v.cpu().numpy())
    h.attrs.update({"backbone": best_ck["backbone"], "img_size": best_ck["img_size"], "mode": cfg["mode"]})
print(f"Best single model: {os.path.basename(best_path)} (val_bal_acc={best_ck['val_bal_acc']:.4f}) -> {h5_path}")
print("Share the whole folder (pipeline_config.json + model_*_fold*.pt): Drive → Share → 'Anyone with the link' → Viewer.")

## 10 · Standalone inference (fresh runtime)
Run §1 (setup + the two `%%writefile` cells + imports) and §2, then this cell. It reads `pipeline_config.json`, so mode / offsets / flip / threshold match training exactly. (Or use `python inference.py` from the repo.)

In [ ]:
def run_inference_only():
    cfg = load_config(CFG.PROJECT_DIR)
    _, _, tm, raw_te = load_data(CFG.PROJECT_DIR, CFG.WORK_DIR, need_train=False)
    X = build_inputs(raw_te, tm["sun_azimuth_angle"].values.astype(np.float64), cfg["mode"], cfg["phi0_deg"])
    p, used = ensemble_predict(CFG.PROJECT_DIR, cfg["runs"], X, cfg["use_vflip"], n_folds=CFG.N_FOLDS)
    s = make_submission(tm, p[:, 1], cfg["threshold"], os.path.join(CFG.PROJECT_DIR, "submission.csv"))
    print(f"Regenerated submission.csv from {len(used)} models"); return s
# run_inference_only()

## 11 · Methodology summary (paste into your submission)

*Every image is rotated counter-clockwise by the sun-azimuth-derived angle (reflect-padded to the image diagonal, rotated, centre-cropped, so no black corners leak class information) so the sun arrives from one fixed direction. Instead of assuming the sign convention, we measure it: a per-image brightness-dipole vector (pointing to the bright side of the object — sun side for a mound, far side for a crater) is fitted against `sun_azimuth_angle` using the training labels. Candidate normalisations (rotate by −az, by +az, both as extra channels, or per-image routing between the two using the image's own shading axis) are compared in a short controlled pilot and the best is used for all training and inference. A constant offset places the light at a canonical angle, which makes a vertical flip label-preserving; no other flips/rotations are used. An ImageNet-21k EfficientNetV2-S is fine-tuned with class-weighted focal loss and EMA in 5-fold stratified CV; fold models are ensembled with physics-safe TTA and a threshold tuned on out-of-fold predictions.*

## 12 · If OOF is still below 95 % — send me these, then try in this order
**Send me:** the §4 table + histograms description, the §5 pilot table, the §7 confusion matrix.
1. **Pilot shows `routed`/`dual` ≫ `spec`** → the mixture theory holds; try `IMG_SIZE = 384`, `EPOCHS = 14`, then the 2nd backbone ensemble (`BACKBONE="convnext_tiny.fb_in22k_ft_in1k"`, `RUN_NAME="convnext"`, `LR=5e-5`, re-run §6 → §7 → §8).
2. **All modes ≈ 0.70–0.75** → the shading↔azimuth relation isn't captured by my estimator (e.g. crops not centred, very low sun elevation, or a different azimuth reference). Check whether the §4 histograms are peaked at all; tell me the values and we'll design the next probe.
3. **Train ≪ val loss gap again** → raise `drop_path`/`weight_decay` in `fit_model`, lower `LR` to `5e-5`.